In [1]:
#!/usr/bin/env python3
"""
generate_scenes.py — Fragmentation Scene Generator
====================================================
Generates Cubit mesh (.g) + Peridigm XML (.xml) pairs for brittle fracture simulations.

Two scene types:
  A) Free-fall: Mug/Vase dropped from ~1.5-2m onto a ground plane
  B) Bullet impact: Bullet fired at a Mug/Vase sitting on a ground plane

Objects: parametric mug (cylinder + handle) or vase (spline-revolved profile)
Material: Brittle elastic with Critical Stretch damage model

Units in Cubit: millimeters (mm) — auto-converted to meters on export.

Material properties (SI units):
    Mug/Vase:  density=2200 kg/m^3, K=14.90e9 Pa, G=8.94e9 Pa
    Bullet:    density=7700 kg/m^3, K=160.0e9 Pa, G=78.3e9 Pa
    Damage:    Critical Stretch = 0.0005

Usage:
    # Adjust CUBIT_PATH below, then:
    python generate_scenes.py
"""

import sys
import os
import random
import math
import json
import xml.etree.ElementTree as ET
from typing import Dict, Optional, Tuple, List
from itertools import combinations
import random as rng

# ============================================================
# ★ EDIT THIS PATH to match your Coreform Cubit installation
# ============================================================
CUBIT_PATH = r"E:\Program Files\Coreform Cubit 2025.12\bin"
sys.path.append(CUBIT_PATH)
import cubit
cubit.init(['cubit', '-nojournal'])
cubit.cmd("reset")

# ============================================================
# Helper: echo to real stdout (Cubit may redirect)
# ============================================================
def echo(msg):
    sys.__stdout__.write(str(msg) + "\n")
    sys.__stdout__.flush()

volume 1: 1849 elements
volume 2: 13558 elements
volume 1: 1849 elements
volume 2: 13558 elements
(30.904550160539703, 25.268069446020316, 56.43876603659086)
volume 1: 1849 elements
volume 2: 13558 elements
volume 1: 1849 elements
volume 2: 13558 elements
(30.904550160539703, 25.268069446020316, 56.43876603659086)
volume 1: 4761 elements
volume 2: 51882 elements
volume 1: 3844 elements
volume 2: 32874 elements
volume 1: 3844 elements
volume 2: 32874 elements
volume 1: 3844 elements
volume 2: 32874 elements
(-17.03319002751061, -17.503678135315614, 35.40970464263282)
volume 1: 1849 elements
volume 2: 13558 elements
volume 3: 48 elements
volume 1: 1849 elements
volume 2: 13558 elements
volume 3: 48 elements
(30.904550160539703, 25.268069446020316, 56.43876603659086)
volume 1: 4761 elements
volume 2: 51882 elements
volume 6: 140 elements
volume 1: 3844 elements
volume 2: 32874 elements
volume 6: 84 elements
volume 1: 3844 elements
volume 2: 32874 elements
volume 6: 84 elements
volume 1: 3

## 1. Geometry Builder Functions
### 1.1. Build Vase Parametric Function 

In [2]:
def make_vase_parametric(
    h_total_mm=170.0, wall_mm=2.0, wall_bottom_mm=3.0,
    r_base_mm=30.0, r_belly_mm=50.0, h_belly_peak_mm=30.0, h_belly_top_mm=85.0,
    r_neck_mm=10.0, h_neck_start_mm=130.0,
    r_mouth_mm=20.0,
    mesh_size_mm=4.0,
):
    h = h_total_mm
    t = wall_mm
    tb = wall_bottom_mm

    outer_pts = [
        (0, 0), (r_base_mm, 0), (r_belly_mm, h_belly_peak_mm),
        (r_base_mm, h_belly_top_mm), (r_neck_mm, h_neck_start_mm), (r_mouth_mm, h),
    ]
    inner_pts = [
        (r_mouth_mm-t, h), (r_neck_mm-t, h_neck_start_mm),
        (r_base_mm-t, h_belly_top_mm), (r_belly_mm-t, h_belly_peak_mm),
        (r_base_mm, tb), (0, tb),
    ]

    all_pts = outer_pts + inner_pts
    vert_ids = []
    for x, z in all_pts:
        cubit.cmd(f"create vertex {x} 0 {z}")
        vert_ids.append(cubit.get_last_id("vertex"))

    n_outer = len(outer_pts)
    curves = []

    cubit.cmd(f"create curve vertex {vert_ids[0]} {vert_ids[1]}")
    curves.append(cubit.get_last_id("curve"))

    cubit.cmd(f"create curve spline vertex {' '.join(str(vert_ids[i]) for i in range(1, n_outer))}")
    curves.append(cubit.get_last_id("curve"))

    cubit.cmd(f"create curve vertex {vert_ids[n_outer-1]} {vert_ids[n_outer]}")
    curves.append(cubit.get_last_id("curve"))

    cubit.cmd(f"create curve spline vertex {' '.join(str(vert_ids[n_outer+i]) for i in range(len(inner_pts)-1))}")
    curves.append(cubit.get_last_id("curve"))

    cubit.cmd(f"create curve vertex {vert_ids[-2]} {vert_ids[-1]}")
    curves.append(cubit.get_last_id("curve"))

    cubit.cmd(f"create curve vertex {vert_ids[-1]} {vert_ids[0]}")
    curves.append(cubit.get_last_id("curve"))

    cubit.cmd(f"create surface curve {' '.join(str(c) for c in curves)}")
    cubit.cmd(f"sweep surface {cubit.get_last_id('surface')} zaxis angle 360")
    vol_id = cubit.get_last_id("volume")

    cubit.cmd(f"volume {vol_id} size {mesh_size_mm}")
    cubit.cmd(f"volume {vol_id} scheme tetmesh")
    cubit.cmd(f"mesh volume {vol_id}")
    return vol_id

In [3]:
cubit.cmd("reset")

vol_id = make_vase_parametric()

cubit.cmd(f"block 1 volume {vol_id}")
out = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\test_vase.g"
cubit.cmd(f'export mesh "{out}" overwrite')

log = out.replace(".g", ".log")
with open(log, "w") as f:
    f.write(f"vol_id: {vol_id}\n")
    f.write(f"nodes: {cubit.get_node_count()}\n")
    f.write(f"elems: {cubit.get_element_count()}\n")
    f.write(f"exists: {os.path.exists(out)}\n")

### 1.2. Build Mug Parametric Function

In [4]:
def make_mug_parametric(
    body_radius_mm: float = 40.0,
    body_height_mm: float = 90.0,
    wall_thickness_mm: float = 4.0,
    handle_width_mm: float = 10.0,
    handle_height_mm: float = 50.0,
    handle_protrusion_mm: float = 25.0,
    mesh_size_mm: float = 4.0,
):
    body_r = body_radius_mm
    body_h = body_height_mm
    wall_t = wall_thickness_mm
    h_w = handle_width_mm
    h_h = handle_height_mm
    h_p = handle_protrusion_mm

    vols_before = set(cubit.get_entities("volume"))

    cubit.cmd(f"create cylinder height {body_h} radius {body_r}")
    outer_cyl = max(set(cubit.get_entities("volume")) - vols_before)
    cubit.cmd(f"move volume {outer_cyl} z {body_h / 2}")

    vols_before2 = set(cubit.get_entities("volume"))
    inner_r = body_r - wall_t
    inner_h = body_h - wall_t
    cubit.cmd(f"create cylinder height {inner_h} radius {inner_r}")
    inner_cyl = max(set(cubit.get_entities("volume")) - vols_before2)
    cubit.cmd(f"move volume {inner_cyl} z {wall_t + inner_h / 2}")

    cubit.cmd(f"subtract volume {inner_cyl} from volume {outer_cyl}")
    body_vol = max(cubit.get_entities("volume"))

    handle_center_x = body_r + h_p / 2
    handle_center_z = body_h * 0.55
    handle_major_r = h_h / 2
    handle_minor_r = h_w / 2

    vols_before3 = set(cubit.get_entities("volume"))
    cubit.cmd(f"create torus major radius {handle_major_r} minor radius {handle_minor_r}")
    torus_vol = max(set(cubit.get_entities("volume")) - vols_before3)

    cubit.cmd(f"rotate volume {torus_vol} angle 90 about x")
    cubit.cmd(f"move volume {torus_vol} x {handle_center_x} z {handle_center_z}")

    vols_before4 = set(cubit.get_entities("volume"))
    cut_size = max(body_h, body_r * 4) * 2
    cubit.cmd(f"brick x {cut_size} y {cut_size} z {cut_size}")
    cut_vol = max(set(cubit.get_entities("volume")) - vols_before4)
    cubit.cmd(f"move volume {cut_vol} x {body_r -wall_t/2 - cut_size / 2} z {handle_center_z}")
    cubit.cmd(f"subtract volume {cut_vol} from volume {torus_vol}")

    all_vols_now = set(cubit.get_entities("volume"))
    handle_vols = all_vols_now - {body_vol} - vols_before
    if handle_vols:
        handle_vol = max(handle_vols)
        cubit.cmd(f"unite volume {body_vol} {handle_vol}")

    final_vol = max(cubit.get_entities("volume"))

    cubit.cmd(f"volume {final_vol} size {mesh_size_mm}")
    cubit.cmd(f"volume {final_vol} scheme tetmesh")
    cubit.cmd(f"mesh volume {final_vol}")

    return final_vol

In [5]:
cubit.cmd("reset")

vol_id = make_mug_parametric()

cubit.cmd(f"block 1 volume {vol_id}")
out = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\test_mug.g"
cubit.cmd(f'export mesh "{out}" overwrite')

log = out.replace(".g", ".log")
with open(log, "w") as f:
    f.write(f"vol_id: {vol_id}\n")
    f.write(f"nodes: {cubit.get_node_count()}\n")
    f.write(f"elems: {cubit.get_element_count()}\n")
    f.write(f"exists: {os.path.exists(out)}\n")

### 1.3. Make Bullet & Floor Parametric

In [6]:
def make_bullet(
    radius_mm: float = 4.0,
    length_mm: float = 15.0,
    mesh_size_mm: float = 2.0,
) -> int:
    cubit.cmd(f"create cylinder height {length_mm} radius {radius_mm}")
    vol_id = cubit.get_last_id("volume")
    assert vol_id > 0, "create cylinder failed"

    cubit.cmd(f"volume {vol_id} size {mesh_size_mm}")
    cubit.cmd(f"volume {vol_id} scheme sweep")
    cubit.cmd(f"mesh volume {vol_id}")
    return vol_id

In [7]:
def make_floor(
    size_x_mm: float = 150.0,
    size_y_mm: float = 150.0,
    thickness_mm: float = 2.0,
    mesh_size_mm: float = 8.0,
) -> int:
    vols_before = set(cubit.get_entities("volume"))
    cubit.cmd(f"brick x {size_x_mm} y {size_y_mm} z {thickness_mm}")
    vol_id = max(set(cubit.get_entities("volume")) - vols_before)
    cubit.cmd(f"move volume {vol_id} z {-thickness_mm / 2}")

    cubit.cmd(f"volume {vol_id} size {mesh_size_mm}")
    cubit.cmd(f"volume {vol_id} scheme sweep")
    cubit.cmd(f"mesh volume {vol_id}")
    return vol_id

## 2. Build Scene

### 2.0. Total Element Count

In [8]:
def total_elements(vol_ids):
    total = 0
    for vid in vol_ids:
        cnt = cubit.get_volume_element_count(vid)
        print(f"volume {vid}: {cnt} elements")
        total += cnt
    return total


### 2.1. Build Vase Drop

In [9]:
import random

def build_vase_drop_scene(seed: int = 42, max_nodes=50000):
    """
    Vase free-fall scene: random vase dropped from 1.5-2m with random orientation.
    All units mm. Scale to m at export.
    
    Block 1 / Nodeset 1: Floor
    Block 2 / Nodeset 2: Vase
    """
    rng = random.Random(seed)
    cubit.cmd("reset")
    r_neck_mm=rng.uniform(10, 20)
    mesh_size = r_neck_mm/4

    # ---- 1. Floor ----
    floor_vol = make_floor(mesh_size_mm=mesh_size)
    cubit.cmd(f"block 1 volume {floor_vol}")
    cubit.cmd("block 1 name 'block_1'")
    cubit.cmd(f"nodeset 1 volume {floor_vol}")
    cubit.cmd("nodeset 1 name 'nodelist_1'")

    # ---- 2. Vase (randomized params, reasonable ranges) ----

    h_total = rng.uniform(120, 200)
    h_belly_peak = rng.uniform(20, 50)
    h_belly_top = rng.uniform(60, 100)
    h_neck_start = rng.uniform(100, 160)

    # 保证 h_belly_peak < h_belly_top < h_neck_start < h_total
    h_belly_top = min(h_belly_top, h_neck_start - 15)
    h_belly_peak = min(h_belly_peak, h_belly_top - 15)
    h_neck_start = min(h_neck_start, h_total - 20)

    r_belly = rng.uniform(35, 65)
    r_base = rng.uniform(20, 40)
    # r_neck = rng.uniform(7, 18)
    r_mouth = rng.uniform(12, 28)
    wall = rng.uniform(1.5, 3.5)

    # 保证内壁半径 > 0
    r_neck_mm = max(r_neck_mm, wall + 2)
    r_mouth = max(r_mouth, wall + 2)
    r_base = max(r_base, wall + 2)

    vase_vol = make_vase_parametric(
        h_total_mm=h_total,
        wall_mm=wall,
        wall_bottom_mm=rng.uniform(2.0, 4.0),
        r_base_mm=r_base,
        r_belly_mm=r_belly,
        h_belly_peak_mm=h_belly_peak,
        h_belly_top_mm=h_belly_top,
        r_neck_mm=r_neck_mm,
        h_neck_start_mm=h_neck_start,
        r_mouth_mm=r_mouth,
        mesh_size_mm=mesh_size,
    )

    # ---- 节点数检查 ----
    debug_path = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\relaxation_debug_vase_drop.log"
    relaxation = 1.0
    with open(debug_path, "w") as dbg:
        dbg.write(f"=== Relaxation loop start ===\n")
        dbg.write(f"floor={floor_vol}, mug={vase_vol}\n")
        dbg.write(f"initial mesh_size={mesh_size}\n")
        
        for attempt in range(10):
            n = total_elements([floor_vol, vase_vol])
            dbg.write(f"attempt={attempt}, elements={n}, max={max_nodes}, relaxation={relaxation}, mesh_size={mesh_size * relaxation}\n")
            dbg.flush()
            
            if n <= max_nodes:
                dbg.write(f"OK: {n} <= {max_nodes}, breaking\n")
                break
            
            relaxation *= 1.1
            mesh_size = mesh_size * relaxation
            dbg.write(f"  remeshing with size={mesh_size}\n")
            
            cubit.cmd(f"delete mesh volume {floor_vol}")
            cubit.cmd(f"delete mesh volume {vase_vol}")
            
            cubit.init(['cubit', '-nojournal'])
            cubit.cmd("reset")
            floor_vol = make_floor(mesh_size_mm=mesh_size)
            cubit.cmd(f"block 1 volume {floor_vol}")
            cubit.cmd("block 1 name 'block_1'")
            cubit.cmd(f"nodeset 1 volume {floor_vol}")
            cubit.cmd("nodeset 1 name 'nodelist_1'")

            # ---- 2. Mug ----
            vase_vol = make_vase_parametric(
                h_total_mm=h_total,
                wall_mm=wall,
                wall_bottom_mm=rng.uniform(2.0, 4.0),
                r_base_mm=r_base,
                r_belly_mm=r_belly,
                h_belly_peak_mm=h_belly_peak,
                h_belly_top_mm=h_belly_top,
                r_neck_mm=r_neck_mm,
                h_neck_start_mm=h_neck_start,
                r_mouth_mm=r_mouth,
                mesh_size_mm=mesh_size,
            )

            n_after = total_elements([floor_vol, vase_vol])
            dbg.write(f"  after remesh: elements={n_after}\n")
            dbg.flush()
        
        dbg.write(f"=== Loop done, final elements={total_elements([floor_vol, vase_vol])} ===\n")
    # ---- 3. Position ----
    drop_height = rng.uniform(1500, 2500)
    xy_offset = 20.0
    dx = rng.uniform(-xy_offset, xy_offset)
    dy = rng.uniform(-xy_offset, xy_offset)

    # 在原点旋转
    rot_x = rng.uniform(-30, 30)
    rot_y = rng.uniform(-30, 30)
    rot_z = rng.uniform(0, 360)
    cubit.cmd(f"rotate volume {vase_vol} angle {rot_z} about z")
    cubit.cmd(f"rotate volume {vase_vol} angle {rot_x} about x")
    cubit.cmd(f"rotate volume {vase_vol} angle {rot_y} about y")
    

    # 旋转后把底部中心挪到原点
    bb = cubit.get_center_point("volume", vase_vol)
    print(bb)
    cx = bb[0]
    cy = bb[1]
    cz = bb[2]
    cubit.cmd(f"move volume {vase_vol} x {-cx} y {-cy} z {-cz}")
    bb_origin = cubit.get_center_point("volume", vase_vol)

    # 再挪到最终位置
    cubit.cmd(f"move volume {vase_vol} x {dx} y {dy} z {drop_height}")

    cubit.cmd(f"block 2 volume {vase_vol}")
    cubit.cmd("block 2 name 'block_2'")
    cubit.cmd(f"nodeset 2 volume {vase_vol}")
    cubit.cmd("nodeset 2 name 'nodelist_2'")

    # ---- 4. Export ----
    cubit.cmd("volume all scale 0.001")

    info = {
        "floor_vol": floor_vol,
        "vase_vol": vase_vol,
        "mesh_size": mesh_size,
        "drop_height_mm": drop_height,
        "rotation": (rot_x, rot_y, rot_z),
        "seed": seed,
        "nodes": cubit.get_node_count(),
        "bounding box after rotation": bb,
        "bounding box after displacement": bb_origin,

    }
    return info

In [10]:
info = build_vase_drop_scene(seed=99, max_nodes=50000)

out = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\vase_drop_test10.g"
cubit.cmd(f'export mesh "{out}" overwrite')

log = out.replace(".g", ".log")
with open(log, "w") as f:
    for k, v in info.items():
        f.write(f"{k}: {v}\n")

In [11]:
from pathlib import Path

def generate_vase_drop_peridigm_xml(
    mesh_file: str,
    info: dict,
    output_xml: str | None = None,
    *,
    floor_block: str = "block_1",
    vase_block: str = "block_2",
    floor_nodeset: str = "nodelist_1",
    vase_nodeset: str = "nodelist_2",
    gravity: float = 9.81,
    verbose: bool = False,
):
    """
    Generate Peridigm XML for vase-drop scene.
    
    Uses info dict from build_vase_drop_scene() to compute:
      - horizon from mesh_size (3.015 * mesh_size_m)
      - dt from CFL condition
      - final_time from drop height
      - output_frequency for ~60 frames
    """

    # ---- Derive physical parameters from info ----
    mesh_size_m = info["mesh_size"] * 0.001  # mm → m
    horizon = 3.015 * mesh_size_m
    drop_height_m = info["drop_height_mm"] * 0.001

    # P-wave speed for ceramic: c_p = sqrt((K + 4G/3) / rho)
    vase_density = 2200.0
    vase_bulk = 14.90e9
    vase_shear = 8.94e9
    c_p_ceramic = math.sqrt((vase_bulk + 4 * vase_shear / 3) / vase_density)  # ~3370 m/s

    floor_density = 7700.0
    floor_bulk = 160.0e9
    floor_shear = 78.3e9
    c_p_steel = math.sqrt((floor_bulk + 4 * floor_shear / 3) / floor_density)  # ~5655 m/s

    # CFL: dt = dx / max(c_p)
    dt = mesh_size_m / max(c_p_ceramic, c_p_steel) * 0.7  # safety factor 0.7

    # Time for free fall: t = sqrt(2h/g) + buffer for bounce/fracture
    t_fall = math.sqrt(2 * drop_height_m / gravity)
    final_time = t_fall * 1.5  # 1.5x buffer

    # Output ~60 frames
    total_steps = int(final_time / dt)
    output_frequency = max(1, total_steps // 60)

    # Contact radius ~ 1.5 * mesh_size
    contact_radius = 1.5 * mesh_size_m
    search_radius = 2.5 * mesh_size_m

    # Damage: Critical Stretch
    critical_stretch = 0.0005

    xml = f'''<?xml version="1.0" encoding="UTF-8"?>
<ParameterList name="Peridigm">

  <ParameterList name="Discretization">
    <Parameter name="Type" type="string" value="Exodus"/>
    <Parameter name="Input Mesh File" type="string" value="{mesh_file}"/>
  </ParameterList>

  <ParameterList name="Materials">
    <ParameterList name="Vase Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{vase_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{vase_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{vase_shear:.6e}"/>
    </ParameterList>
    <ParameterList name="Floor Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{floor_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{floor_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{floor_shear:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Damage Models">
    <ParameterList name="Vase Damage">
      <Parameter name="Damage Model" type="string" value="Critical Stretch"/>
      <Parameter name="Critical Stretch" type="double" value="{critical_stretch}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Blocks">
    <ParameterList name="Floor Block">
      <Parameter name="Block Names" type="string" value="{floor_block}"/>
      <Parameter name="Material" type="string" value="Floor Material"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
    <ParameterList name="Vase Block">
      <Parameter name="Block Names" type="string" value="{vase_block}"/>
      <Parameter name="Material" type="string" value="Vase Material"/>
      <Parameter name="Damage Model" type="string" value="Vase Damage"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Contact">
    <Parameter name="Search Radius" type="double" value="{search_radius:.6e}"/>
    <Parameter name="Search Frequency" type="int" value="100"/>
    <ParameterList name="Models">
      <ParameterList name="Vase Floor Contact">
        <Parameter name="Contact Model" type="string" value="Short Range Force"/>
        <Parameter name="Contact Radius" type="double" value="{contact_radius:.6e}"/>
        <Parameter name="Spring Constant" type="double" value="1.0e12"/>
      </ParameterList>
    </ParameterList>
    <ParameterList name="Interactions">
      <ParameterList name="Interaction Vase Floor">
        <Parameter name="First Block" type="string" value="{vase_block}"/>
        <Parameter name="Second Block" type="string" value="{floor_block}"/>
        <Parameter name="Contact Model" type="string" value="Vase Floor Contact"/>
      </ParameterList>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Boundary Conditions">
    <ParameterList name="Fix Floor X">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="x"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Fix Floor Y">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="y"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Fix Floor Z">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Gravity Vase">
      <Parameter name="Type" type="string" value="Body Force"/>
      <Parameter name="Node Set" type="string" value="{vase_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="-{gravity*vase_density}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Solver">
    <Parameter name="Verbose" type="bool" value="{str(verbose).lower()}"/>
    <Parameter name="Initial Time" type="double" value="0.0"/>
    <Parameter name="Final Time" type="double" value="{final_time:.6e}"/>
    <ParameterList name="Verlet">
      <Parameter name="Fixed dt" type="double" value="{dt:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Output">
    <Parameter name="Output File Type" type="string" value="ExodusII"/>
    <Parameter name="Output Filename" type="string" value="{Path(mesh_file).stem}"/>
    <Parameter name="Output Frequency" type="int" value="{output_frequency}"/>
    <ParameterList name="Output Variables">
      <Parameter name="Displacement" type="bool" value="true"/>
      <Parameter name="Velocity" type="bool" value="true"/>
      <Parameter name="Element_Id" type="bool" value="true"/>
      <Parameter name="Proc_Num" type="bool" value="true"/>
      <Parameter name="Dilatation" type="bool" value="true"/>
      <Parameter name="Weighted_Volume" type="bool" value="true"/>
      <Parameter name="Volume" type="bool" value="true"/>
      <Parameter name="Force" type="bool" value="true"/>
      <Parameter name="Contact_Force" type="bool" value="true"/>
      <Parameter name="Number_Of_Neighbors" type="bool" value="true"/>
      <Parameter name="Radius" type="bool" value="true"/>
      <Parameter name="Coordinates" type="bool" value="true"/>
      <Parameter name="Force_Density" type="bool" value="true"/>
      <Parameter name="Kinetic_Energy" type="bool" value="true"/>
    </ParameterList>
  </ParameterList>

</ParameterList>
'''

    if output_xml is not None:
        Path(output_xml).write_text(xml, encoding="utf-8")

    return xml

In [12]:
info = build_vase_drop_scene(seed=99, max_nodes=50000)

mesh_out = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\vase_drop_test.g"
cubit.cmd(f'export mesh "{mesh_out}" dimension 3 overwrite')

xml_out = mesh_out.replace(".g", ".xml")
xml = generate_vase_drop_peridigm_xml(
    mesh_file=os.path.basename(mesh_out),
    info=info,
    output_xml=xml_out,
)

### 2.2. Build Mug Drop

In [13]:
def build_mug_drop_scene(seed: int = 42, max_nodes: int = 50000):
    rng = random.Random(seed)
    cubit.cmd("reset")
    body_radius_mm = rng.uniform(30, 50)
    body_height_mm = rng.uniform(70, 115)
    wall_thickness_mm = rng.uniform(2.5, 5.0)
    handle_width_mm = rng.uniform(7, 14)
    handle_height_mm = rng.uniform(35, 60)
    handle_protrusion_mm = rng.uniform(18, 35)
    mesh_size = handle_width_mm / 4
    

    # ---- 1. Floor ----
    floor_vol = make_floor(mesh_size_mm=mesh_size)
    cubit.cmd(f"block 1 volume {floor_vol}")
    cubit.cmd("block 1 name 'block_1'")
    cubit.cmd(f"nodeset 1 volume {floor_vol}")
    cubit.cmd("nodeset 1 name 'nodelist_1'")

    # ---- 2. Mug ----
    mug_vol = make_mug_parametric(
        body_radius_mm=body_radius_mm,
        body_height_mm=body_height_mm,
        wall_thickness_mm=wall_thickness_mm,
        handle_width_mm=handle_width_mm,
        handle_height_mm=handle_height_mm,
        handle_protrusion_mm=handle_protrusion_mm,
        mesh_size_mm=mesh_size,
    )

    # ---- 节点数检查 ----
    debug_path = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\relaxation_debug_mug_drop.log"
    relaxation = 1.0
    with open(debug_path, "w") as dbg:
        dbg.write(f"=== Relaxation loop start ===\n")
        dbg.write(f"floor={floor_vol}, mug={mug_vol}\n")
        dbg.write(f"initial mesh_size={mesh_size}\n")
        
        for attempt in range(10):
            n = total_elements([floor_vol, mug_vol])
            dbg.write(f"attempt={attempt}, elements={n}, max={max_nodes}, relaxation={relaxation}, mesh_size={mesh_size * relaxation}\n")
            dbg.flush()
            
            if n <= max_nodes:
                dbg.write(f"OK: {n} <= {max_nodes}, breaking\n")
                break
            
            relaxation *= 1.1
            mesh_size_new = mesh_size * relaxation
            dbg.write(f"  remeshing with size={mesh_size_new}\n")
            
            cubit.cmd(f"delete mesh volume {floor_vol}")
            cubit.cmd(f"delete mesh volume {mug_vol}")
            
            cubit.init(['cubit', '-nojournal'])
            cubit.cmd("reset")
            floor_vol = make_floor(mesh_size_mm=mesh_size_new)
            cubit.cmd(f"block 1 volume {floor_vol}")
            cubit.cmd("block 1 name 'block_1'")
            cubit.cmd(f"nodeset 1 volume {floor_vol}")
            cubit.cmd("nodeset 1 name 'nodelist_1'")

            # ---- 2. Mug ----
            mug_vol = make_mug_parametric(
                body_radius_mm=body_radius_mm,
                body_height_mm=body_height_mm,
                wall_thickness_mm=wall_thickness_mm,
                handle_width_mm=handle_width_mm,
                handle_height_mm=handle_height_mm,
                handle_protrusion_mm=handle_protrusion_mm,
                mesh_size_mm=mesh_size_new,
            )

            n_after = total_elements([floor_vol, mug_vol])
            dbg.write(f"  after remesh: elements={n_after}\n")
            dbg.flush()
        
        dbg.write(f"=== Loop done, final elements={total_elements([floor_vol, mug_vol])} ===\n")

    # ---- 3. Position ----
    drop_height = rng.uniform(1500, 2500)
    xy_offset = 20.0
    dx = rng.uniform(-xy_offset, xy_offset)
    dy = rng.uniform(-xy_offset, xy_offset)

    # 在原点旋转
    rot_x = rng.uniform(-30, 30)
    rot_y = rng.uniform(-30, 30)
    rot_z = rng.uniform(0, 360)
    cubit.cmd(f"rotate volume {mug_vol} angle {rot_z} about z")
    cubit.cmd(f"rotate volume {mug_vol} angle {rot_x} about x")
    cubit.cmd(f"rotate volume {mug_vol} angle {rot_y} about y")
    

    # 旋转后把底部中心挪到原点
    bb = cubit.get_center_point("volume", mug_vol)
    print(bb)
    cx = bb[0]
    cy = bb[1]
    cz = bb[2]
    cubit.cmd(f"move volume {mug_vol} x {-cx} y {-cy} z {-cz}")
    bb_origin = cubit.get_center_point("volume", mug_vol)

    # 再挪到最终位置
    cubit.cmd(f"move volume {mug_vol} x {dx} y {dy} z {drop_height}")

    cubit.cmd(f"block 2 volume {mug_vol}")
    cubit.cmd("block 2 name 'block_2'")
    cubit.cmd(f"nodeset 2 volume {mug_vol}")
    cubit.cmd("nodeset 2 name 'nodelist_2'")

    # ---- 4. Export ----
    cubit.cmd("volume all scale 0.001")

    info = {
        "floor_vol": floor_vol,
        "vase_vol": mug_vol,
        "mesh_size": mesh_size,
        "drop_height_mm": drop_height,
        "rotation": (rot_x, rot_y, rot_z),
        "seed": seed,
        "nodes": cubit.get_node_count(),
        "bounding box after rotation": bb,
        "bounding box after displacement": bb_origin,

    }
    return info

In [14]:
def generate_mug_drop_peridigm_xml(
    mesh_file: str,
    info: dict,
    output_xml: str | None = None,
    *,
    floor_block: str = "block_1",
    mug_block: str = "block_2",
    floor_nodeset: str = "nodelist_1",
    mug_nodeset: str = "nodelist_2",
    gravity: float = 9.81,
    verbose: bool = False,
):
    mesh_size_m = info["mesh_size"] * 0.001
    horizon = 3.015 * mesh_size_m

    mug_density = 2200.0
    mug_bulk = 14.90e9
    mug_shear = 8.94e9
    c_p_ceramic = math.sqrt((mug_bulk + 4 * mug_shear / 3) / mug_density)

    floor_density = 7700.0
    floor_bulk = 160.0e9
    floor_shear = 78.3e9
    c_p_steel = math.sqrt((floor_bulk + 4 * floor_shear / 3) / floor_density)

    dt = mesh_size_m / max(c_p_ceramic, c_p_steel) * 0.7

    final_time = 2.0
    fps = 60
    total_steps = int(final_time / dt)
    output_frequency = max(1, total_steps // (int(final_time * fps)))

    contact_radius = 1.5 * mesh_size_m
    search_radius = 2.5 * mesh_size_m
    critical_stretch = 0.0005

    xml = f'''<?xml version="1.0" encoding="UTF-8"?>
<ParameterList name="Peridigm">

  <ParameterList name="Discretization">
    <Parameter name="Type" type="string" value="Exodus"/>
    <Parameter name="Input Mesh File" type="string" value="{mesh_file}"/>
  </ParameterList>

  <ParameterList name="Materials">
    <ParameterList name="Mug Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{mug_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{mug_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{mug_shear:.6e}"/>
    </ParameterList>
    <ParameterList name="Floor Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{floor_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{floor_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{floor_shear:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Damage Models">
    <ParameterList name="Mug Damage">
      <Parameter name="Damage Model" type="string" value="Critical Stretch"/>
      <Parameter name="Critical Stretch" type="double" value="{critical_stretch}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Blocks">
    <ParameterList name="Floor Block">
      <Parameter name="Block Names" type="string" value="{floor_block}"/>
      <Parameter name="Material" type="string" value="Floor Material"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
    <ParameterList name="Mug Block">
      <Parameter name="Block Names" type="string" value="{mug_block}"/>
      <Parameter name="Material" type="string" value="Mug Material"/>
      <Parameter name="Damage Model" type="string" value="Mug Damage"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Contact">
    <Parameter name="Search Radius" type="double" value="{search_radius:.6e}"/>
    <Parameter name="Search Frequency" type="int" value="100"/>
    <ParameterList name="Models">
      <ParameterList name="Mug Floor Contact">
        <Parameter name="Contact Model" type="string" value="Short Range Force"/>
        <Parameter name="Contact Radius" type="double" value="{contact_radius:.6e}"/>
        <Parameter name="Spring Constant" type="double" value="1.0e12"/>
      </ParameterList>
    </ParameterList>
    <ParameterList name="Interactions">
      <ParameterList name="Interaction Mug Floor">
        <Parameter name="First Block" type="string" value="{mug_block}"/>
        <Parameter name="Second Block" type="string" value="{floor_block}"/>
        <Parameter name="Contact Model" type="string" value="Mug Floor Contact"/>
      </ParameterList>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Boundary Conditions">
    <ParameterList name="Fix Floor X">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="x"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Fix Floor Y">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="y"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Fix Floor Z">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Gravity Mug">
      <Parameter name="Type" type="string" value="Body Force"/>
      <Parameter name="Node Set" type="string" value="{mug_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{-mug_density * gravity:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Solver">
    <Parameter name="Verbose" type="bool" value="{str(verbose).lower()}"/>
    <Parameter name="Initial Time" type="double" value="0.0"/>
    <Parameter name="Final Time" type="double" value="{final_time:.6e}"/>
    <ParameterList name="Verlet">
      <Parameter name="Fixed dt" type="double" value="{dt:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Output">
    <Parameter name="Output File Type" type="string" value="ExodusII"/>
    <Parameter name="Output Filename" type="string" value="{Path(mesh_file).stem}"/>
    <Parameter name="Output Frequency" type="int" value="{output_frequency}"/>
    <ParameterList name="Output Variables">
      <Parameter name="Displacement" type="bool" value="true"/>
      <Parameter name="Velocity" type="bool" value="true"/>
      <Parameter name="Element_Id" type="bool" value="true"/>
      <Parameter name="Proc_Num" type="bool" value="true"/>
      <Parameter name="Dilatation" type="bool" value="true"/>
      <Parameter name="Weighted_Volume" type="bool" value="true"/>
      <Parameter name="Volume" type="bool" value="true"/>
      <Parameter name="Force" type="bool" value="true"/>
      <Parameter name="Contact_Force" type="bool" value="true"/>
      <Parameter name="Number_Of_Neighbors" type="bool" value="true"/>
      <Parameter name="Radius" type="bool" value="true"/>
      <Parameter name="Coordinates" type="bool" value="true"/>
      <Parameter name="Force_Density" type="bool" value="true"/>
      <Parameter name="Kinetic_Energy" type="bool" value="true"/>
      <Parameter name="Damage" type="bool" value="true"/>
    </ParameterList>
  </ParameterList>

</ParameterList>
'''

    if output_xml is not None:
        Path(output_xml).write_text(xml, encoding="utf-8")

    return xml

In [15]:
info = build_mug_drop_scene(seed=99, max_nodes=50000)

mesh_out = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\mug_drop_test.g"
cubit.cmd(f'export mesh "{mesh_out}" dimension 3 overwrite')

xml_out = mesh_out.replace(".g", ".xml")
xml = generate_mug_drop_peridigm_xml(
    mesh_file=os.path.basename(mesh_out),
    info=info,
    output_xml=xml_out,
)

log = mesh_out.replace(".g", ".log")
with open(log, "w") as f:
    for k, v in info.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\ndt: {info['mesh_size'] * 0.001 / 5655 * 0.7:.6e}\n")
    f.write(f"horizon: {3.015 * info['mesh_size'] * 0.001:.6e}\n")
    f.write(f"xml: {xml_out}\n")

### 2.3. Bullet Vase Scene

In [16]:
def build_bullet_vase_scene(seed: int = 42, max_nodes=50000):

    """
    Bullet impact on vase: vase sits on floor, bullet fires from random direction.
    Bullet speed 100-300 m/s, oriented along flight direction.
    All units mm. Scale to m at export.

    Block 1 / Nodeset 1: Floor
    Block 2 / Nodeset 2: Vase
    Block 3 / Nodeset 3: Bullet
    """
    rng = random.Random(seed)
    cubit.cmd("reset")
    r_neck_mm=rng.uniform(10, 20)
    mesh_size = r_neck_mm/4
    debug_path = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\relaxation_debug_bullet_vase.log"
    # ---- 0. Parameters ----
    h_total = rng.uniform(120, 200)
    h_belly_peak = rng.uniform(20, 50)
    h_belly_top = rng.uniform(60, 100)
    h_neck_start = rng.uniform(100, 160)

    # 保证 h_belly_peak < h_belly_top < h_neck_start < h_total
    h_belly_top = min(h_belly_top, h_neck_start - 15)
    h_belly_peak = min(h_belly_peak, h_belly_top - 15)
    h_neck_start = min(h_neck_start, h_total - 20)

    r_belly = rng.uniform(35, 65)
    r_base = rng.uniform(20, 40)
    # r_neck = rng.uniform(7, 18)
    r_mouth = rng.uniform(12, 28)
    wall = rng.uniform(1.5, 3.5)

    # 保证内壁半径 > 0
    r_neck_mm = max(r_neck_mm, wall + 2)
    r_mouth = max(r_mouth, wall + 2)
    r_base = max(r_base, wall + 2)

    # ---- 1. Floor ----
    floor_vol = make_floor(mesh_size_mm=mesh_size)
    cubit.cmd(f"block 1 volume {floor_vol}")
    cubit.cmd("block 1 name 'block_1'")
    cubit.cmd(f"nodeset 1 volume {floor_vol}")
    cubit.cmd("nodeset 1 name 'nodelist_1'")

    # ---- 2. Vase (randomized params, reasonable ranges) ----
    vase_vol = make_vase_parametric(
        h_total_mm=h_total,
        wall_mm=wall,
        wall_bottom_mm=rng.uniform(2.0, 4.0),
        r_base_mm=r_base,
        r_belly_mm=r_belly,
        h_belly_peak_mm=h_belly_peak,
        h_belly_top_mm=h_belly_top,
        r_neck_mm=r_neck_mm,
        h_neck_start_mm=h_neck_start,
        r_mouth_mm=r_mouth,
        mesh_size_mm=mesh_size,
    )
    # ---- 3. Bullet ----
    bullet_vol = make_bullet(mesh_size_mm=mesh_size)

    # ---- 节点数检查，超过 max_nodes 则放大 mesh size 重新划分 ----
    relaxation = 1.0
    with open(debug_path, "w") as dbg:
        dbg.write(f"=== Relaxation loop start ===\n")
        dbg.write(f"floor={floor_vol}, vase={vase_vol}, bullet={bullet_vol}\n")
        dbg.write(f"initial mesh_size={mesh_size}\n")
        
        for attempt in range(10):
            n = total_elements([floor_vol, vase_vol, bullet_vol])
            dbg.write(f"attempt={attempt}, elements={n}, max={max_nodes}, relaxation={relaxation}, mesh_size={mesh_size * relaxation}\n")
            dbg.flush()
            
            if n <= max_nodes:
                dbg.write(f"OK: {n} <= {max_nodes}, breaking\n")
                break
            
            relaxation *= 1.1
            mesh_size_new = mesh_size * relaxation
            dbg.write(f"  remeshing with size={mesh_size_new}\n")
            
            cubit.cmd(f"delete mesh volume {floor_vol}")
            cubit.cmd(f"delete mesh volume {vase_vol}")
            cubit.cmd(f"delete mesh volume {bullet_vol}")
            
            cubit.init(['cubit', '-nojournal'])
            cubit.cmd("reset")
            floor_vol = make_floor(mesh_size_mm=mesh_size_new)
            cubit.cmd(f"block 1 volume {floor_vol}")
            cubit.cmd("block 1 name 'block_1'")
            cubit.cmd(f"nodeset 1 volume {floor_vol}")
            cubit.cmd("nodeset 1 name 'nodelist_1'")

            # ---- 2. Mug ----
            vase_vol = make_vase_parametric(
                h_total_mm=h_total,
                wall_mm=wall,
                wall_bottom_mm=rng.uniform(2.0, 4.0),
                r_base_mm=r_base,
                r_belly_mm=r_belly,
                h_belly_peak_mm=h_belly_peak,
                h_belly_top_mm=h_belly_top,
                r_neck_mm=r_neck_mm,
                h_neck_start_mm=h_neck_start,
                r_mouth_mm=r_mouth,
                mesh_size_mm=mesh_size,
            )
            # ---- 3. Bullet ----
            bullet_vol = make_bullet(mesh_size_mm=mesh_size_new)

            n_after = total_elements([floor_vol, vase_vol, bullet_vol])
            dbg.write(f"  after remesh: elements={n_after}\n")
            dbg.flush()
        
        dbg.write(f"=== Loop done, final elements={total_elements([floor_vol, vase_vol, bullet_vol])} ===\n")

    # ---- 4. Position ----
    bullet_dist = 200
    xyz_offset = 15.0
    x_off = rng.uniform(-xyz_offset, xyz_offset)
    y_off = rng.uniform(-xyz_offset, xyz_offset)
    z_off = rng.uniform(-xyz_offset, xyz_offset)

    # 在原点旋转
    rot_x = rng.uniform(-30, 30)
    rot_y = rng.uniform(-30, 30)
    rot_z = rng.uniform(0, 360)
    cubit.cmd(f"rotate volume {vase_vol} angle {rot_z} about z")
    cubit.cmd(f"rotate volume {vase_vol} angle {rot_x} about x")
    cubit.cmd(f"rotate volume {vase_vol} angle {rot_y} about y")
    # 旋转后把底部中心挪到原点
    bb = cubit.get_center_point("volume", vase_vol)
    boundingbox = cubit.get_bounding_box("volume", vase_vol)
    print(bb)
    cx = bb[0]
    cy = bb[1]
    cz = boundingbox[6]
    cubit.cmd(f"move volume {vase_vol} x {-cx} y {-cy} z {-cz+2}")
    bb_origin = cubit.get_center_point("volume", vase_vol)

    # 子弹放到中心点
    bulletpos = cubit.get_center_point("volume", bullet_vol)
    cubit.cmd(f"move volume {bullet_vol} x {-bulletpos[0]} y {-bulletpos[1]} z {-bulletpos[2]}")

    # 在原点旋转
    rot_x = rng.uniform(-30, 30)
    rot_y = rng.uniform(-30, 30)
    rot_z = rng.uniform(0, 360)
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_z} about z")
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_x} about x")
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_y} about y")

    # 初始方向 (1, 0, 0)，依次绕 Z、X、Y 旋转
    rx = math.radians(rot_x)
    ry = math.radians(rot_y)
    rz = math.radians(rot_z)

    # 绕 Z
    dx = 0.0
    dy = 0.0
    dz = 1.0

    # 绕 X
    dy2 = dy * math.cos(rx) - dz * math.sin(rx)
    dz2 = dy * math.sin(rx) + dz * math.cos(rx)
    dy, dz = dy2, dz2

    # 绕 Y
    dx2 = dx * math.cos(ry) + dz * math.sin(ry)
    dz2 = -dx * math.sin(ry) + dz * math.cos(ry)
    dx, dz = dx2, dz2
    # (dx, dy, dz) 就是子弹飞行方向的单位向量
    # 子弹起点：花瓶中心反方向 1500mm
    dx = dx * bullet_dist
    dy = dy * bullet_dist
    dz = dz * bullet_dist

    # 先挪到击中花瓶的位置
    cx = bb_origin[0]+x_off
    cy = bb_origin[1]+y_off
    cz = bb_origin[2]+z_off
    cubit.cmd(f"move volume {bullet_vol} x {cx} y {cy} z {cz}")
    with open(r"F:\Peridigm-pre-process\dynamic-impact-generation\files\debug.log", "w") as f:
        f.write(f"bb_origin: {bb_origin}\n")
        f.write(f"cx={cx}, cy={cy}, cz={cz}\n")
        f.write(f"x_off={x_off}, y_off={y_off}, z_off={z_off}\n")


    # 再沿着圆柱中轴线飞走
    cubit.cmd(f"move volume {bullet_vol} x {dx} y {dy} z {dz}")

    cubit.cmd(f"block 2 volume {vase_vol}")
    cubit.cmd("block 2 name 'block_2'")
    cubit.cmd(f"nodeset 2 volume {vase_vol}")
    cubit.cmd("nodeset 2 name 'nodelist_2'")

    cubit.cmd(f"block 3 volume {bullet_vol}")
    cubit.cmd("block 3 name 'block_3'")
    cubit.cmd(f"nodeset 3 volume {bullet_vol}")
    cubit.cmd("nodeset 3 name 'nodelist_3'")

    # ---- 4. Export ----
    cubit.cmd("volume all scale 0.001")

    cubit.cmd("delete free vertex all")
    cubit.cmd("delete free curve all")
    cubit.cmd("delete free surface all")

    info = {
        "floor_vol": floor_vol,
        "vase_vol": vase_vol,
        "bullet_vol": bullet_vol,
        "rotation": (rot_x, rot_y, rot_z),
        "seed": seed,
        "elements": n,
        "mesh_relaxation": relaxation,
        "dx": dx / bullet_dist, 
        "dy": dy / bullet_dist,
        "dz": dz / bullet_dist,
        "mesh_size": mesh_size,
    }
    return info

    



In [17]:
def generate_bullet_vase_peridigm_xml(
    mesh_file: str,
    info: dict,
    output_xml: str | None = None,
    *,
    floor_block: str = "block_1",
    vase_block: str = "block_2",
    bullet_block: str = "block_3",
    floor_nodeset: str = "nodelist_1",
    vase_nodeset: str = "nodelist_2",
    bullet_nodeset: str = "nodelist_3",
    bullet_speed: float = 400.0,
    gravity: float = 9.81,
    verbose: bool = False,
):
    mesh_size_m = info["mesh_size"] * 0.001
    horizon = 3.015 * mesh_size_m

    vase_density = 2200.0
    vase_bulk = 14.90e9
    vase_shear = 8.94e9

    floor_density = 7700.0
    floor_bulk = 160.0e9
    floor_shear = 78.3e9

    bullet_density = 7700.0
    bullet_bulk = 160.0e9
    bullet_shear = 78.3e9

    c_p_ceramic = math.sqrt((vase_bulk + 4 * vase_shear / 3) / vase_density)
    c_p_steel = math.sqrt((floor_bulk + 4 * floor_shear / 3) / floor_density)

    dt = mesh_size_m / max(c_p_ceramic, c_p_steel) * 0.7

    # Bullet scene: short duration, 0.05s enough for impact + fragmentation
    final_time = 0.05
    fps = 60
    total_steps = int(final_time / dt)
    output_frequency = max(1, total_steps // int(final_time * fps))

    contact_radius = 1.5 * mesh_size_m
    search_radius = 2.5 * mesh_size_m
    critical_stretch = 0.0005

    # Bullet velocity components (m/s) — direction already unit vector, scale by speed
    vx = -info["dx"] * bullet_speed
    vy = -info["dy"] * bullet_speed
    vz = -info["dz"] * bullet_speed

    xml = f'''<?xml version="1.0" encoding="UTF-8"?>
<ParameterList name="Peridigm">

  <ParameterList name="Discretization">
    <Parameter name="Type" type="string" value="Exodus"/>
    <Parameter name="Input Mesh File" type="string" value="{mesh_file}"/>
  </ParameterList>

  <ParameterList name="Materials">
    <ParameterList name="Vase Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{vase_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{vase_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{vase_shear:.6e}"/>
    </ParameterList>
    <ParameterList name="Floor Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{floor_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{floor_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{floor_shear:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{bullet_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{bullet_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{bullet_shear:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Damage Models">
    <ParameterList name="Vase Damage">
      <Parameter name="Damage Model" type="string" value="Critical Stretch"/>
      <Parameter name="Critical Stretch" type="double" value="{critical_stretch}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Blocks">
    <ParameterList name="Floor Block">
      <Parameter name="Block Names" type="string" value="{floor_block}"/>
      <Parameter name="Material" type="string" value="Floor Material"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
    <ParameterList name="Vase Block">
      <Parameter name="Block Names" type="string" value="{vase_block}"/>
      <Parameter name="Material" type="string" value="Vase Material"/>
      <Parameter name="Damage Model" type="string" value="Vase Damage"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Block">
      <Parameter name="Block Names" type="string" value="{bullet_block}"/>
      <Parameter name="Material" type="string" value="Bullet Material"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Contact">
    <Parameter name="Search Radius" type="double" value="{search_radius:.6e}"/>
    <Parameter name="Search Frequency" type="int" value="100"/>
    <ParameterList name="Models">
      <ParameterList name="Bullet Vase Contact">
        <Parameter name="Contact Model" type="string" value="Short Range Force"/>
        <Parameter name="Contact Radius" type="double" value="{contact_radius:.6e}"/>
        <Parameter name="Spring Constant" type="double" value="1.0e13"/>
      </ParameterList>
      <ParameterList name="Vase Floor Contact">
        <Parameter name="Contact Model" type="string" value="Short Range Force"/>
        <Parameter name="Contact Radius" type="double" value="{contact_radius:.6e}"/>
        <Parameter name="Spring Constant" type="double" value="1.0e12"/>
      </ParameterList>
      <ParameterList name="Bullet Floor Contact">
        <Parameter name="Contact Model" type="string" value="Short Range Force"/>
        <Parameter name="Contact Radius" type="double" value="{contact_radius:.6e}"/>
        <Parameter name="Spring Constant" type="double" value="1.0e13"/>
      </ParameterList>
    </ParameterList>
    <ParameterList name="Interactions">
      <ParameterList name="Interaction Bullet Vase">
        <Parameter name="First Block" type="string" value="{bullet_block}"/>
        <Parameter name="Second Block" type="string" value="{vase_block}"/>
        <Parameter name="Contact Model" type="string" value="Bullet Vase Contact"/>
      </ParameterList>
      <ParameterList name="Interaction Vase Floor">
        <Parameter name="First Block" type="string" value="{vase_block}"/>
        <Parameter name="Second Block" type="string" value="{floor_block}"/>
        <Parameter name="Contact Model" type="string" value="Vase Floor Contact"/>
      </ParameterList>
      <ParameterList name="Interaction Bullet Floor">
        <Parameter name="First Block" type="string" value="{bullet_block}"/>
        <Parameter name="Second Block" type="string" value="{floor_block}"/>
        <Parameter name="Contact Model" type="string" value="Bullet Floor Contact"/>
      </ParameterList>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Boundary Conditions">
    <ParameterList name="Fix Floor X">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="x"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Fix Floor Y">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="y"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Fix Floor Z">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Gravity Vase">
      <Parameter name="Type" type="string" value="Body Force"/>
      <Parameter name="Node Set" type="string" value="{vase_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{-vase_density * gravity:.6e}"/>
    </ParameterList>
    <ParameterList name="Gravity Bullet">
      <Parameter name="Type" type="string" value="Body Force"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{-bullet_density * gravity:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity X">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="x"/>
      <Parameter name="Value" type="string" value="{vx:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity Y">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="y"/>
      <Parameter name="Value" type="string" value="{vy:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity Z">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{vz:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Solver">
    <Parameter name="Verbose" type="bool" value="{str(verbose).lower()}"/>
    <Parameter name="Initial Time" type="double" value="0.0"/>
    <Parameter name="Final Time" type="double" value="{final_time:.6e}"/>
    <ParameterList name="Verlet">
      <Parameter name="Fixed dt" type="double" value="{dt:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Output">
    <Parameter name="Output File Type" type="string" value="ExodusII"/>
    <Parameter name="Output Filename" type="string" value="{Path(mesh_file).stem}"/>
    <Parameter name="Output Frequency" type="int" value="{output_frequency}"/>
    <ParameterList name="Output Variables">
      <Parameter name="Displacement" type="bool" value="true"/>
      <Parameter name="Velocity" type="bool" value="true"/>
      <Parameter name="Element_Id" type="bool" value="true"/>
      <Parameter name="Proc_Num" type="bool" value="true"/>
      <Parameter name="Dilatation" type="bool" value="true"/>
      <Parameter name="Weighted_Volume" type="bool" value="true"/>
      <Parameter name="Volume" type="bool" value="true"/>
      <Parameter name="Force" type="bool" value="true"/>
      <Parameter name="Contact_Force" type="bool" value="true"/>
      <Parameter name="Number_Of_Neighbors" type="bool" value="true"/>
      <Parameter name="Radius" type="bool" value="true"/>
      <Parameter name="Coordinates" type="bool" value="true"/>
      <Parameter name="Force_Density" type="bool" value="true"/>
      <Parameter name="Kinetic_Energy" type="bool" value="true"/>
      <Parameter name="Damage" type="bool" value="true"/>
    </ParameterList>
  </ParameterList>

</ParameterList>
'''

    if output_xml is not None:
        Path(output_xml).write_text(xml, encoding="utf-8")

    return xml

In [18]:
info = build_bullet_vase_scene(seed=99, max_nodes=50000)

mesh_out = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\bullet_vase_test.g"
cubit.cmd(f'export mesh "{mesh_out}" dimension 3 overwrite')

xml_out = mesh_out.replace(".g", ".xml")
xml = generate_bullet_vase_peridigm_xml(
    mesh_file=os.path.basename(mesh_out),
    info=info,
    output_xml=xml_out,
    bullet_speed=rng.uniform(200, 500),
)

log = mesh_out.replace(".g", ".log")
with open(log, "w") as f:
    for k, v in info.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nxml: {xml_out}\n")

### 2.3. Bullet Mug Scene

In [19]:
def build_bullet_mug_scene(seed: int = 42, max_nodes=30000):

    """
    Bullet impact on vase: vase sits on floor, bullet fires from random direction.
    Bullet speed 100-300 m/s, oriented along flight direction.
    All units mm. Scale to m at export.

    Block 1 / Nodeset 1: Floor
    Block 2 / Nodeset 2: Vase
    Block 3 / Nodeset 3: Bullet
    """
    rng = random.Random(seed)
    cubit.cmd("reset")
    body_radius_mm = rng.uniform(30, 50)
    debug_path = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\relaxation_debug_bullet_mug.log"

    
    body_height_mm = rng.uniform(70, 115)
    wall_thickness_mm = rng.uniform(2.5, 5.0)
    handle_width_mm = rng.uniform(7, 14)
    handle_height_mm = rng.uniform(35, 60)
    handle_protrusion_mm = rng.uniform(18, 35)
    mesh_size = handle_width_mm / 4

    # ---- 1. Floor ----
    floor_vol = make_floor(mesh_size_mm=mesh_size)
    cubit.cmd(f"block 1 volume {floor_vol}")
    cubit.cmd("block 1 name 'block_1'")
    cubit.cmd(f"nodeset 1 volume {floor_vol}")
    cubit.cmd("nodeset 1 name 'nodelist_1'")

    # ---- 2. Mug ----
    mug_vol = make_mug_parametric(
        body_radius_mm=body_radius_mm,
        body_height_mm=body_height_mm,
        wall_thickness_mm=wall_thickness_mm,
        handle_width_mm=handle_width_mm,
        handle_height_mm=handle_height_mm,
        handle_protrusion_mm=handle_protrusion_mm,
        mesh_size_mm=mesh_size,
    )
    # ---- 3. Bullet ----
    bullet_vol = make_bullet(mesh_size_mm=mesh_size)

    # ---- 节点数检查 ----
    relaxation = 1.0
    with open(debug_path, "w") as dbg:
        dbg.write(f"=== Relaxation loop start ===\n")
        dbg.write(f"floor={floor_vol}, mug={mug_vol}, bullet={bullet_vol}\n")
        dbg.write(f"initial mesh_size={mesh_size}\n")
        
        for attempt in range(10):
            n = total_elements([floor_vol, mug_vol, bullet_vol])
            dbg.write(f"attempt={attempt}, elements={n}, max={max_nodes}, relaxation={relaxation}, mesh_size={mesh_size * relaxation}\n")
            dbg.flush()
            
            if n <= max_nodes:
                dbg.write(f"OK: {n} <= {max_nodes}, breaking\n")
                break
            
            relaxation *= 1.1
            mesh_size_new = mesh_size * relaxation
            dbg.write(f"  remeshing with size={mesh_size_new}\n")
            
            cubit.cmd(f"delete mesh volume {floor_vol}")
            cubit.cmd(f"delete mesh volume {mug_vol}")
            cubit.cmd(f"delete mesh volume {bullet_vol}")
            
            cubit.init(['cubit', '-nojournal'])
            cubit.cmd("reset")
            floor_vol = make_floor(mesh_size_mm=mesh_size_new)
            cubit.cmd(f"block 1 volume {floor_vol}")
            cubit.cmd("block 1 name 'block_1'")
            cubit.cmd(f"nodeset 1 volume {floor_vol}")
            cubit.cmd("nodeset 1 name 'nodelist_1'")

            # ---- 2. Mug ----
            mug_vol = make_mug_parametric(
                body_radius_mm=body_radius_mm,
                body_height_mm=body_height_mm,
                wall_thickness_mm=wall_thickness_mm,
                handle_width_mm=handle_width_mm,
                handle_height_mm=handle_height_mm,
                handle_protrusion_mm=handle_protrusion_mm,
                mesh_size_mm=mesh_size_new,
            )
            # ---- 3. Bullet ----
            bullet_vol = make_bullet(mesh_size_mm=mesh_size_new)

            n_after = total_elements([floor_vol, mug_vol, bullet_vol])
            dbg.write(f"  after remesh: elements={n_after}\n")
            dbg.flush()
        
        dbg.write(f"=== Loop done, final elements={total_elements([floor_vol, mug_vol, bullet_vol])} ===\n")

    # ---- 4. Position ----
    bullet_dist = 200
    xyz_offset = 15.0
    x_off = rng.uniform(-xyz_offset, xyz_offset)
    y_off = rng.uniform(-xyz_offset, xyz_offset)
    z_off = rng.uniform(-xyz_offset, xyz_offset)

    # 在原点旋转
    rot_x = rng.uniform(-30, 30)
    rot_y = rng.uniform(-30, 30)
    rot_z = rng.uniform(0, 360)
    cubit.cmd(f"rotate volume {mug_vol} angle {rot_z} about z")
    cubit.cmd(f"rotate volume {mug_vol} angle {rot_x} about x")
    cubit.cmd(f"rotate volume {mug_vol} angle {rot_y} about y")

    # 旋转后把底部中心挪到原点
    bb = cubit.get_center_point("volume", mug_vol)
    boundingbox = cubit.get_bounding_box("volume", mug_vol)
    print(bb)
    cx = bb[0]
    cy = bb[1]
    cz = boundingbox[6]
    cubit.cmd(f"move volume {mug_vol} x {-cx} y {-cy} z {-cz+2}")
    bb_origin = cubit.get_center_point("volume", mug_vol)
    

    # 子弹放到中心点
    bulletpos = cubit.get_center_point("volume", bullet_vol)
    cubit.cmd(f"move volume {bullet_vol} x {-bulletpos[0]} y {-bulletpos[1]} z {-bulletpos[2]}")

    # 在原点旋转
    rot_x = rng.uniform(-30, 30)
    rot_y = rng.uniform(-30, 30)
    rot_z = rng.uniform(0, 360)
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_z} about z")
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_x} about x")
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_y} about y")

    # 初始方向 (1, 0, 0)，依次绕 Z、X、Y 旋转
    rx = math.radians(rot_x)
    ry = math.radians(rot_y)
    rz = math.radians(rot_z)

    # 绕 Z
    dx = 0.0
    dy = 0.0
    dz = 1.0

    # 绕 X
    dy2 = dy * math.cos(rx) - dz * math.sin(rx)
    dz2 = dy * math.sin(rx) + dz * math.cos(rx)
    dy, dz = dy2, dz2

    # 绕 Y
    dx2 = dx * math.cos(ry) + dz * math.sin(ry)
    dz2 = -dx * math.sin(ry) + dz * math.cos(ry)
    dx, dz = dx2, dz2
    # (dx, dy, dz) 就是子弹飞行方向的单位向量
    # 子弹起点：花瓶中心反方向 1500mm
    dx = dx * bullet_dist
    dy = dy * bullet_dist
    dz = dz * bullet_dist

    # 先挪到击中花瓶的位置
    cx = bb_origin[0]+x_off
    cy = bb_origin[1]+y_off
    cz = bb_origin[2]+z_off
    cubit.cmd(f"move volume {bullet_vol} x {cx} y {cy} z {cz}")
    with open(r"F:\Peridigm-pre-process\dynamic-impact-generation\files\debug.log", "w") as f:
        f.write(f"bb_origin: {bb_origin}\n")
        f.write(f"cx={cx}, cy={cy}, cz={cz}\n")
        f.write(f"x_off={x_off}, y_off={y_off}, z_off={z_off}\n")


    # 再沿着圆柱中轴线飞走
    cubit.cmd(f"move volume {bullet_vol} x {dx} y {dy} z {dz}")

    cubit.cmd(f"block 2 volume {mug_vol}")
    cubit.cmd("block 2 name 'block_2'")
    cubit.cmd(f"nodeset 2 volume {mug_vol}")
    cubit.cmd("nodeset 2 name 'nodelist_2'")

    cubit.cmd(f"block 3 volume {bullet_vol}")
    cubit.cmd("block 3 name 'block_3'")
    cubit.cmd(f"nodeset 3 volume {bullet_vol}")
    cubit.cmd("nodeset 3 name 'nodelist_3'")

    # ---- 4. Export ----
    cubit.cmd("volume all scale 0.001")

    cubit.cmd("delete free vertex all")
    cubit.cmd("delete free curve all")
    cubit.cmd("delete free surface all")

    info = {
        "floor_vol": floor_vol,
        "vase_vol": mug_vol,
        "bullet_vol": bullet_vol,
        "rotation": (rot_x, rot_y, rot_z),
        "seed": seed,
        "elements": n,
        "mesh_relaxation": relaxation,
        "dx": dx / bullet_dist, 
        "dy": dy / bullet_dist,
        "dz": dz / bullet_dist,
        "mesh_size": mesh_size,
    }
    return info

In [20]:
def generate_bullet_mug_peridigm_xml(
    mesh_file: str,
    info: dict,
    output_xml: str | None = None,
    *,
    floor_block: str = "block_1",
    mug_block: str = "block_2",
    bullet_block: str = "block_3",
    floor_nodeset: str = "nodelist_1",
    mug_nodeset: str = "nodelist_2",
    bullet_nodeset: str = "nodelist_3",
    bullet_speed: float = 400.0,
    gravity: float = 9.81,
    verbose: bool = False,
):
    mesh_size_m = info["mesh_size"] * 0.001
    horizon = 3.015 * mesh_size_m

    mug_density = 2200.0
    mug_bulk = 14.90e9
    mug_shear = 8.94e9

    floor_density = 7700.0
    floor_bulk = 160.0e9
    floor_shear = 78.3e9

    bullet_density = 7700.0
    bullet_bulk = 160.0e9
    bullet_shear = 78.3e9

    c_p_ceramic = math.sqrt((mug_bulk + 4 * mug_shear / 3) / mug_density)
    c_p_steel = math.sqrt((floor_bulk + 4 * floor_shear / 3) / floor_density)

    dt = mesh_size_m / max(c_p_ceramic, c_p_steel) * 0.7

    final_time = 0.05
    fps = 60
    total_steps = int(final_time / dt)
    output_frequency = max(1, total_steps // int(final_time * fps))

    contact_radius = 1.5 * mesh_size_m
    search_radius = 2.5 * mesh_size_m
    critical_stretch = 0.0005

    vx = -info["dx"] * bullet_speed
    vy = -info["dy"] * bullet_speed
    vz = -info["dz"] * bullet_speed

    xml = f'''<?xml version="1.0" encoding="UTF-8"?>
<ParameterList name="Peridigm">

  <ParameterList name="Discretization">
    <Parameter name="Type" type="string" value="Exodus"/>
    <Parameter name="Input Mesh File" type="string" value="{mesh_file}"/>
  </ParameterList>

  <ParameterList name="Materials">
    <ParameterList name="Mug Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{mug_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{mug_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{mug_shear:.6e}"/>
    </ParameterList>
    <ParameterList name="Floor Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{floor_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{floor_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{floor_shear:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{bullet_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{bullet_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{bullet_shear:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Damage Models">
    <ParameterList name="Mug Damage">
      <Parameter name="Damage Model" type="string" value="Critical Stretch"/>
      <Parameter name="Critical Stretch" type="double" value="{critical_stretch}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Blocks">
    <ParameterList name="Floor Block">
      <Parameter name="Block Names" type="string" value="{floor_block}"/>
      <Parameter name="Material" type="string" value="Floor Material"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
    <ParameterList name="Mug Block">
      <Parameter name="Block Names" type="string" value="{mug_block}"/>
      <Parameter name="Material" type="string" value="Mug Material"/>
      <Parameter name="Damage Model" type="string" value="Mug Damage"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Block">
      <Parameter name="Block Names" type="string" value="{bullet_block}"/>
      <Parameter name="Material" type="string" value="Bullet Material"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Contact">
    <Parameter name="Search Radius" type="double" value="{search_radius:.6e}"/>
    <Parameter name="Search Frequency" type="int" value="100"/>
    <ParameterList name="Models">
      <ParameterList name="Bullet Mug Contact">
        <Parameter name="Contact Model" type="string" value="Short Range Force"/>
        <Parameter name="Contact Radius" type="double" value="{contact_radius:.6e}"/>
        <Parameter name="Spring Constant" type="double" value="1.0e13"/>
      </ParameterList>
      <ParameterList name="Mug Floor Contact">
        <Parameter name="Contact Model" type="string" value="Short Range Force"/>
        <Parameter name="Contact Radius" type="double" value="{contact_radius:.6e}"/>
        <Parameter name="Spring Constant" type="double" value="1.0e12"/>
      </ParameterList>
      <ParameterList name="Bullet Floor Contact">
        <Parameter name="Contact Model" type="string" value="Short Range Force"/>
        <Parameter name="Contact Radius" type="double" value="{contact_radius:.6e}"/>
        <Parameter name="Spring Constant" type="double" value="1.0e13"/>
      </ParameterList>
    </ParameterList>
    <ParameterList name="Interactions">
      <ParameterList name="Interaction Bullet Mug">
        <Parameter name="First Block" type="string" value="{bullet_block}"/>
        <Parameter name="Second Block" type="string" value="{mug_block}"/>
        <Parameter name="Contact Model" type="string" value="Bullet Mug Contact"/>
      </ParameterList>
      <ParameterList name="Interaction Mug Floor">
        <Parameter name="First Block" type="string" value="{mug_block}"/>
        <Parameter name="Second Block" type="string" value="{floor_block}"/>
        <Parameter name="Contact Model" type="string" value="Mug Floor Contact"/>
      </ParameterList>
      <ParameterList name="Interaction Bullet Floor">
        <Parameter name="First Block" type="string" value="{bullet_block}"/>
        <Parameter name="Second Block" type="string" value="{floor_block}"/>
        <Parameter name="Contact Model" type="string" value="Bullet Floor Contact"/>
      </ParameterList>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Boundary Conditions">
    <ParameterList name="Fix Floor X">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="x"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Fix Floor Y">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="y"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Fix Floor Z">
      <Parameter name="Type" type="string" value="Prescribed Displacement"/>
      <Parameter name="Node Set" type="string" value="{floor_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="0.0"/>
    </ParameterList>
    <ParameterList name="Gravity Mug">
      <Parameter name="Type" type="string" value="Body Force"/>
      <Parameter name="Node Set" type="string" value="{mug_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{-mug_density * gravity:.6e}"/>
    </ParameterList>
    <ParameterList name="Gravity Bullet">
      <Parameter name="Type" type="string" value="Body Force"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{-bullet_density * gravity:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity X">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="x"/>
      <Parameter name="Value" type="string" value="{vx:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity Y">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="y"/>
      <Parameter name="Value" type="string" value="{vy:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity Z">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{vz:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Solver">
    <Parameter name="Verbose" type="bool" value="{str(verbose).lower()}"/>
    <Parameter name="Initial Time" type="double" value="0.0"/>
    <Parameter name="Final Time" type="double" value="{final_time:.6e}"/>
    <ParameterList name="Verlet">
      <Parameter name="Fixed dt" type="double" value="{dt:.6e}"/>
    </ParameterList>
  </ParameterList>

  <ParameterList name="Output">
    <Parameter name="Output File Type" type="string" value="ExodusII"/>
    <Parameter name="Output Filename" type="string" value="{Path(mesh_file).stem}"/>
    <Parameter name="Output Frequency" type="int" value="{output_frequency}"/>
    <ParameterList name="Output Variables">
      <Parameter name="Displacement" type="bool" value="true"/>
      <Parameter name="Velocity" type="bool" value="true"/>
      <Parameter name="Element_Id" type="bool" value="true"/>
      <Parameter name="Proc_Num" type="bool" value="true"/>
      <Parameter name="Dilatation" type="bool" value="true"/>
      <Parameter name="Weighted_Volume" type="bool" value="true"/>
      <Parameter name="Volume" type="bool" value="true"/>
      <Parameter name="Force" type="bool" value="true"/>
      <Parameter name="Contact_Force" type="bool" value="true"/>
      <Parameter name="Number_Of_Neighbors" type="bool" value="true"/>
      <Parameter name="Radius" type="bool" value="true"/>
      <Parameter name="Coordinates" type="bool" value="true"/>
      <Parameter name="Force_Density" type="bool" value="true"/>
      <Parameter name="Kinetic_Energy" type="bool" value="true"/>
      <Parameter name="Damage" type="bool" value="true"/>
    </ParameterList>
  </ParameterList>

</ParameterList>
'''

    if output_xml is not None:
        Path(output_xml).write_text(xml, encoding="utf-8")

    return xml

In [21]:
info = build_bullet_mug_scene(seed=99, max_nodes=50000)

mesh_out = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\bullet_mug_test.g"
cubit.cmd(f'export mesh "{mesh_out}" dimension 3 overwrite')

xml_out = mesh_out.replace(".g", ".xml")
xml = generate_bullet_mug_peridigm_xml(
    mesh_file=os.path.basename(mesh_out),
    info=info,
    output_xml=xml_out,
    bullet_speed=random.uniform(100, 300),
)

log = mesh_out.replace(".g", ".log")
with open(log, "w") as f:
    for k, v in info.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nxml: {xml_out}\n")

# Bullet with No Floor

In [22]:
def build_bullet_vase_nf_scene(seed: int = 42, max_nodes=50000):
    """
    Bullet impact on vase WITHOUT floor. Vase floats at origin.
    All units mm. Scale to m at export.

    Block 1 / Nodeset 1: Vase
    Block 2 / Nodeset 2: Bullet
    """
    rng = random.Random(seed)
    cubit.cmd("reset")
    r_neck_mm = rng.uniform(10, 20)
    mesh_size = r_neck_mm / 4
    debug_path = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\relaxation_debug_bullet_vase_nf.log"

    h_total = rng.uniform(120, 200)
    h_belly_peak = rng.uniform(20, 50)
    h_belly_top = rng.uniform(60, 100)
    h_neck_start = rng.uniform(100, 160)
    h_neck_start = min(h_neck_start, h_total - 20)
    h_belly_top = min(h_belly_top, h_neck_start - 15)
    h_belly_peak = min(h_belly_peak, h_belly_top - 15)

    r_belly = rng.uniform(35, 65)
    r_base = rng.uniform(20, 40)
    r_mouth = rng.uniform(12, 28)
    wall = rng.uniform(1.5, 3.5)
    r_neck_mm = max(r_neck_mm, wall + 2)
    r_mouth = max(r_mouth, wall + 2)
    r_base = max(r_base, wall + 2)

    # ---- 1. Vase ----
    vase_vol = make_vase_parametric(
        h_total_mm=h_total, wall_mm=wall,
        wall_bottom_mm=rng.uniform(2.0, 4.0),
        r_base_mm=r_base, r_belly_mm=r_belly,
        h_belly_peak_mm=h_belly_peak, h_belly_top_mm=h_belly_top,
        r_neck_mm=r_neck_mm, h_neck_start_mm=h_neck_start,
        r_mouth_mm=r_mouth, mesh_size_mm=mesh_size,
    )
    # ---- 2. Bullet ----
    bullet_vol = make_bullet(mesh_size_mm=mesh_size)

    # ---- 节点数检查 ----
    relaxation = 1.0
    with open(debug_path, "w") as dbg:
        dbg.write(f"=== Relaxation loop start ===\n")
        dbg.write(f"vase={vase_vol}, bullet={bullet_vol}\n")
        dbg.write(f"initial mesh_size={mesh_size}\n")
        for attempt in range(10):
            n = total_elements([vase_vol, bullet_vol])
            dbg.write(f"attempt={attempt}, elements={n}, max={max_nodes}, relaxation={relaxation}, mesh_size={mesh_size * relaxation}\n")
            dbg.flush()
            if n <= max_nodes:
                dbg.write(f"OK: {n} <= {max_nodes}, breaking\n")
                break
            relaxation *= 1.1
            mesh_size_new = mesh_size * relaxation
            dbg.write(f"  remeshing with size={mesh_size_new}\n")
            cubit.init(['cubit', '-nojournal'])
            cubit.cmd("reset")
            vase_vol = make_vase_parametric(
                h_total_mm=h_total, wall_mm=wall,
                wall_bottom_mm=rng.uniform(2.0, 4.0),
                r_base_mm=r_base, r_belly_mm=r_belly,
                h_belly_peak_mm=h_belly_peak, h_belly_top_mm=h_belly_top,
                r_neck_mm=r_neck_mm, h_neck_start_mm=h_neck_start,
                r_mouth_mm=r_mouth, mesh_size_mm=mesh_size_new,
            )
            bullet_vol = make_bullet(mesh_size_mm=mesh_size_new)
            n_after = total_elements([vase_vol, bullet_vol])
            dbg.write(f"  after remesh: elements={n_after}\n")
            dbg.flush()
        dbg.write(f"=== Loop done, final elements={total_elements([vase_vol, bullet_vol])} ===\n")

    # ---- 3. Position ----
    bullet_dist = 200
    xyz_offset = 15.0
    x_off = rng.uniform(-xyz_offset, xyz_offset)
    y_off = rng.uniform(-xyz_offset, xyz_offset)
    z_off = rng.uniform(-xyz_offset, xyz_offset)

    # Vase: center at origin, no rotation (bullet scene, vase stationary)
    bb = cubit.get_center_point("volume", vase_vol)
    boundingbox = cubit.get_bounding_box("volume", vase_vol)
    cubit.cmd(f"move volume {vase_vol} x {-bb[0]} y {-bb[1]} z {-boundingbox[6] + 2}")
    bb_origin = cubit.get_center_point("volume", vase_vol)

    # Bullet: center to origin, rotate, compute direction, move out
    bulletpos = cubit.get_center_point("volume", bullet_vol)
    cubit.cmd(f"move volume {bullet_vol} x {-bulletpos[0]} y {-bulletpos[1]} z {-bulletpos[2]}")

    rot_x = rng.uniform(-30, 30)
    rot_y = rng.uniform(-30, 30)
    rot_z = rng.uniform(0, 360)
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_z} about z")
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_x} about x")
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_y} about y")

    rx = math.radians(rot_x)
    ry = math.radians(rot_y)
    rz = math.radians(rot_z)
    dx, dy, dz = 0.0, 0.0, 1.0
    dy2 = dy * math.cos(rx) - dz * math.sin(rx)
    dz2 = dy * math.sin(rx) + dz * math.cos(rx)
    dy, dz = dy2, dz2
    dx2 = dx * math.cos(ry) + dz * math.sin(ry)
    dz2 = -dx * math.sin(ry) + dz * math.cos(ry)
    dx, dz = dx2, dz2

    cx = bb_origin[0] + x_off
    cy = bb_origin[1] + y_off
    cz = bb_origin[2] + z_off
    cubit.cmd(f"move volume {bullet_vol} x {cx} y {cy} z {cz}")
    cubit.cmd(f"move volume {bullet_vol} x {dx * bullet_dist} y {dy * bullet_dist} z {dz * bullet_dist}")

    cubit.cmd(f"block 1 volume {vase_vol}")
    cubit.cmd("block 1 name 'block_1'")
    cubit.cmd(f"nodeset 1 volume {vase_vol}")
    cubit.cmd("nodeset 1 name 'nodelist_1'")

    cubit.cmd(f"block 2 volume {bullet_vol}")
    cubit.cmd("block 2 name 'block_2'")
    cubit.cmd(f"nodeset 2 volume {bullet_vol}")
    cubit.cmd("nodeset 2 name 'nodelist_2'")

    cubit.cmd("volume all scale 0.001")
    cubit.cmd("delete free vertex all")
    cubit.cmd("delete free curve all")
    cubit.cmd("delete free surface all")

    info = {
        "vase_vol": vase_vol, "bullet_vol": bullet_vol,
        "rotation": (rot_x, rot_y, rot_z), "seed": seed,
        "elements": n, "mesh_relaxation": relaxation,
        "dx": dx, "dy": dy, "dz": dz, "mesh_size": mesh_size,
    }
    return info

In [ ]:
def generate_bullet_vase_nf_peridigm_xml(
    mesh_file: str, info: dict, output_xml: str | None = None, *,
    vase_block: str = "block_1", bullet_block: str = "block_2",
    vase_nodeset: str = "nodelist_1", bullet_nodeset: str = "nodelist_2",
    bullet_speed: float = 400.0, gravity: float = 9.81, verbose: bool = False,
):
    mesh_size_m = info["mesh_size"] * 0.001
    horizon = 3.015 * mesh_size_m
    vase_density, vase_bulk, vase_shear = 2200.0, 14.90e9, 8.94e9
    bullet_density, bullet_bulk, bullet_shear = 7700.0, 160.0e9, 78.3e9
    c_p = max(
        math.sqrt((vase_bulk + 4*vase_shear/3) / vase_density),
        math.sqrt((bullet_bulk + 4*bullet_shear/3) / bullet_density),
    )
    dt = mesh_size_m / c_p * 0.7
    final_time = 0.005
    total_steps = int(final_time / dt)
    output_frequency = max(1, total_steps // int(final_time * 100000))
    contact_radius = 1.5 * mesh_size_m
    search_radius = 2.5 * mesh_size_m
    vx = -info["dx"] * bullet_speed
    vy = -info["dy"] * bullet_speed
    vz = -info["dz"] * bullet_speed

    xml = f'''<?xml version="1.0" encoding="UTF-8"?>
<ParameterList name="Peridigm">
  <ParameterList name="Discretization">
    <Parameter name="Type" type="string" value="Exodus"/>
    <Parameter name="Input Mesh File" type="string" value="{mesh_file}"/>
  </ParameterList>
  <ParameterList name="Materials">
    <ParameterList name="Vase Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{vase_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{vase_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{vase_shear:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{bullet_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{bullet_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{bullet_shear:.6e}"/>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Damage Models">
    <ParameterList name="Vase Damage">
      <Parameter name="Damage Model" type="string" value="Critical Stretch"/>
      <Parameter name="Critical Stretch" type="double" value="0.0005"/>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Blocks">
    <ParameterList name="Vase Block">
      <Parameter name="Block Names" type="string" value="{vase_block}"/>
      <Parameter name="Material" type="string" value="Vase Material"/>
      <Parameter name="Damage Model" type="string" value="Vase Damage"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Block">
      <Parameter name="Block Names" type="string" value="{bullet_block}"/>
      <Parameter name="Material" type="string" value="Bullet Material"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Contact">
    <Parameter name="Search Radius" type="double" value="{search_radius:.6e}"/>
    <Parameter name="Search Frequency" type="int" value="100"/>
    <ParameterList name="Models">
      <ParameterList name="Bullet Vase Contact">
        <Parameter name="Contact Model" type="string" value="Short Range Force"/>
        <Parameter name="Contact Radius" type="double" value="{contact_radius:.6e}"/>
        <Parameter name="Spring Constant" type="double" value="1.0e13"/>
      </ParameterList>
    </ParameterList>
    <ParameterList name="Interactions">
      <ParameterList name="Interaction Bullet Vase">
        <Parameter name="First Block" type="string" value="{bullet_block}"/>
        <Parameter name="Second Block" type="string" value="{vase_block}"/>
        <Parameter name="Contact Model" type="string" value="Bullet Vase Contact"/>
      </ParameterList>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Boundary Conditions">
    <ParameterList name="Gravity Vase">
      <Parameter name="Type" type="string" value="Body Force"/>
      <Parameter name="Node Set" type="string" value="{vase_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{-vase_density * gravity:.6e}"/>
    </ParameterList>
    <ParameterList name="Gravity Bullet">
      <Parameter name="Type" type="string" value="Body Force"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{-bullet_density * gravity:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity X">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="x"/>
      <Parameter name="Value" type="string" value="{vx:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity Y">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="y"/>
      <Parameter name="Value" type="string" value="{vy:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity Z">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{vz:.6e}"/>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Solver">
    <Parameter name="Verbose" type="bool" value="{str(verbose).lower()}"/>
    <Parameter name="Initial Time" type="double" value="0.0"/>
    <Parameter name="Final Time" type="double" value="{final_time:.6e}"/>
    <ParameterList name="Verlet">
      <Parameter name="Fixed dt" type="double" value="{dt:.6e}"/>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Output">
    <Parameter name="Output File Type" type="string" value="ExodusII"/>
    <Parameter name="Output Filename" type="string" value="{Path(mesh_file).stem}"/>
    <Parameter name="Output Frequency" type="int" value="{output_frequency}"/>
    <ParameterList name="Output Variables">
      <Parameter name="Displacement" type="bool" value="true"/>
      <Parameter name="Velocity" type="bool" value="true"/>
      <Parameter name="Element_Id" type="bool" value="true"/>
      <Parameter name="Proc_Num" type="bool" value="true"/>
      <Parameter name="Dilatation" type="bool" value="true"/>
      <Parameter name="Weighted_Volume" type="bool" value="true"/>
      <Parameter name="Volume" type="bool" value="true"/>
      <Parameter name="Force" type="bool" value="true"/>
      <Parameter name="Contact_Force" type="bool" value="true"/>
      <Parameter name="Number_Of_Neighbors" type="bool" value="true"/>
      <Parameter name="Radius" type="bool" value="true"/>
      <Parameter name="Coordinates" type="bool" value="true"/>
      <Parameter name="Force_Density" type="bool" value="true"/>
      <Parameter name="Kinetic_Energy" type="bool" value="true"/>
      <Parameter name="Damage" type="bool" value="true"/>
    </ParameterList>
  </ParameterList>
</ParameterList>
'''
    if output_xml is not None:
        Path(output_xml).write_text(xml, encoding="utf-8")
    return xml

In [24]:
def build_bullet_mug_nf_scene(seed: int = 42, max_nodes=50000):
    """
    Bullet impact on mug WITHOUT floor. Mug floats at origin.
    All units mm. Scale to m at export.

    Block 1 / Nodeset 1: Mug
    Block 2 / Nodeset 2: Bullet
    """
    rng = random.Random(seed)
    cubit.cmd("reset")
    debug_path = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\relaxation_debug_bullet_mug_nf.log"

    body_radius_mm = rng.uniform(30, 50)
    body_height_mm = rng.uniform(70, 115)
    wall_thickness_mm = rng.uniform(2.5, 5.0)
    handle_width_mm = rng.uniform(7, 14)
    handle_height_mm = rng.uniform(35, 60)
    handle_protrusion_mm = rng.uniform(18, 35)
    mesh_size = handle_width_mm / 4

    # ---- 1. Mug ----
    mug_vol = make_mug_parametric(
        body_radius_mm=body_radius_mm, body_height_mm=body_height_mm,
        wall_thickness_mm=wall_thickness_mm, handle_width_mm=handle_width_mm,
        handle_height_mm=handle_height_mm, handle_protrusion_mm=handle_protrusion_mm,
        mesh_size_mm=mesh_size,
    )
    # ---- 2. Bullet ----
    bullet_vol = make_bullet(mesh_size_mm=mesh_size)

    # ---- 节点数检查 ----
    relaxation = 1.0
    with open(debug_path, "w") as dbg:
        dbg.write(f"=== Relaxation loop start ===\n")
        dbg.write(f"mug={mug_vol}, bullet={bullet_vol}\n")
        dbg.write(f"initial mesh_size={mesh_size}\n")
        for attempt in range(10):
            n = total_elements([mug_vol, bullet_vol])
            dbg.write(f"attempt={attempt}, elements={n}, max={max_nodes}, relaxation={relaxation}, mesh_size={mesh_size * relaxation}\n")
            dbg.flush()
            if n <= max_nodes:
                dbg.write(f"OK: {n} <= {max_nodes}, breaking\n")
                break
            relaxation *= 1.1
            mesh_size_new = mesh_size * relaxation
            dbg.write(f"  remeshing with size={mesh_size_new}\n")
            cubit.init(['cubit', '-nojournal'])
            cubit.cmd("reset")
            mug_vol = make_mug_parametric(
                body_radius_mm=body_radius_mm, body_height_mm=body_height_mm,
                wall_thickness_mm=wall_thickness_mm, handle_width_mm=handle_width_mm,
                handle_height_mm=handle_height_mm, handle_protrusion_mm=handle_protrusion_mm,
                mesh_size_mm=mesh_size_new,
            )
            bullet_vol = make_bullet(mesh_size_mm=mesh_size_new)
            n_after = total_elements([mug_vol, bullet_vol])
            dbg.write(f"  after remesh: elements={n_after}\n")
            dbg.flush()
        dbg.write(f"=== Loop done, final elements={total_elements([mug_vol, bullet_vol])} ===\n")

    # ---- 3. Position ----
    bullet_dist = 200
    xyz_offset = 15.0
    x_off = rng.uniform(-xyz_offset, xyz_offset)
    y_off = rng.uniform(-xyz_offset, xyz_offset)
    z_off = rng.uniform(-xyz_offset, xyz_offset)

    # Mug at origin
    bb = cubit.get_center_point("volume", mug_vol)
    boundingbox = cubit.get_bounding_box("volume", mug_vol)
    cubit.cmd(f"move volume {mug_vol} x {-bb[0]} y {-bb[1]} z {-boundingbox[6] + 2}")
    bb_origin = cubit.get_center_point("volume", mug_vol)

    # Bullet
    bulletpos = cubit.get_center_point("volume", bullet_vol)
    cubit.cmd(f"move volume {bullet_vol} x {-bulletpos[0]} y {-bulletpos[1]} z {-bulletpos[2]}")

    rot_x = rng.uniform(-30, 30)
    rot_y = rng.uniform(-30, 30)
    rot_z = rng.uniform(0, 360)
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_z} about z")
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_x} about x")
    cubit.cmd(f"rotate volume {bullet_vol} angle {rot_y} about y")

    rx = math.radians(rot_x)
    ry = math.radians(rot_y)
    dx, dy, dz = 0.0, 0.0, 1.0
    dy2 = dy * math.cos(rx) - dz * math.sin(rx)
    dz2 = dy * math.sin(rx) + dz * math.cos(rx)
    dy, dz = dy2, dz2
    dx2 = dx * math.cos(ry) + dz * math.sin(ry)
    dz2 = -dx * math.sin(ry) + dz * math.cos(ry)
    dx, dz = dx2, dz2

    cx = bb_origin[0] + x_off
    cy = bb_origin[1] + y_off
    cz = bb_origin[2] + z_off
    cubit.cmd(f"move volume {bullet_vol} x {cx} y {cy} z {cz}")
    cubit.cmd(f"move volume {bullet_vol} x {dx * bullet_dist} y {dy * bullet_dist} z {dz * bullet_dist}")

    cubit.cmd(f"block 1 volume {mug_vol}")
    cubit.cmd("block 1 name 'block_1'")
    cubit.cmd(f"nodeset 1 volume {mug_vol}")
    cubit.cmd("nodeset 1 name 'nodelist_1'")

    cubit.cmd(f"block 2 volume {bullet_vol}")
    cubit.cmd("block 2 name 'block_2'")
    cubit.cmd(f"nodeset 2 volume {bullet_vol}")
    cubit.cmd("nodeset 2 name 'nodelist_2'")

    cubit.cmd("volume all scale 0.001")
    cubit.cmd("delete free vertex all")
    cubit.cmd("delete free curve all")
    cubit.cmd("delete free surface all")

    info = {
        "mug_vol": mug_vol, "bullet_vol": bullet_vol,
        "rotation": (rot_x, rot_y, rot_z), "seed": seed,
        "elements": n, "mesh_relaxation": relaxation,
        "dx": dx, "dy": dy, "dz": dz, "mesh_size": mesh_size,
    }
    return info

In [ ]:
def generate_bullet_mug_nf_peridigm_xml(
    mesh_file: str, info: dict, output_xml: str | None = None, *,
    mug_block: str = "block_1", bullet_block: str = "block_2",
    mug_nodeset: str = "nodelist_1", bullet_nodeset: str = "nodelist_2",
    bullet_speed: float = 400.0, gravity: float = 9.81, verbose: bool = False,
):
    mesh_size_m = info["mesh_size"] * 0.001
    horizon = 3.015 * mesh_size_m
    mug_density, mug_bulk, mug_shear = 2200.0, 14.90e9, 8.94e9
    bullet_density, bullet_bulk, bullet_shear = 7700.0, 160.0e9, 78.3e9
    c_p = max(
        math.sqrt((mug_bulk + 4*mug_shear/3) / mug_density),
        math.sqrt((bullet_bulk + 4*bullet_shear/3) / bullet_density),
    )
    dt = mesh_size_m / c_p * 0.7
    final_time = 0.05
    total_steps = int(final_time / dt)
    output_frequency = max(1, total_steps // int(final_time * 100000))
    contact_radius = 1.5 * mesh_size_m
    search_radius = 2.5 * mesh_size_m
    vx = -info["dx"] * bullet_speed
    vy = -info["dy"] * bullet_speed
    vz = -info["dz"] * bullet_speed

    xml = f'''<?xml version="1.0" encoding="UTF-8"?>
<ParameterList name="Peridigm">
  <ParameterList name="Discretization">
    <Parameter name="Type" type="string" value="Exodus"/>
    <Parameter name="Input Mesh File" type="string" value="{mesh_file}"/>
  </ParameterList>
  <ParameterList name="Materials">
    <ParameterList name="Mug Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{mug_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{mug_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{mug_shear:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Material">
      <Parameter name="Material Model" type="string" value="Elastic"/>
      <Parameter name="Density" type="double" value="{bullet_density}"/>
      <Parameter name="Bulk Modulus" type="double" value="{bullet_bulk:.6e}"/>
      <Parameter name="Shear Modulus" type="double" value="{bullet_shear:.6e}"/>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Damage Models">
    <ParameterList name="Mug Damage">
      <Parameter name="Damage Model" type="string" value="Critical Stretch"/>
      <Parameter name="Critical Stretch" type="double" value="0.0005"/>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Blocks">
    <ParameterList name="Mug Block">
      <Parameter name="Block Names" type="string" value="{mug_block}"/>
      <Parameter name="Material" type="string" value="Mug Material"/>
      <Parameter name="Damage Model" type="string" value="Mug Damage"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Block">
      <Parameter name="Block Names" type="string" value="{bullet_block}"/>
      <Parameter name="Material" type="string" value="Bullet Material"/>
      <Parameter name="Horizon" type="double" value="{horizon:.6e}"/>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Contact">
    <Parameter name="Search Radius" type="double" value="{search_radius:.6e}"/>
    <Parameter name="Search Frequency" type="int" value="100"/>
    <ParameterList name="Models">
      <ParameterList name="Bullet Mug Contact">
        <Parameter name="Contact Model" type="string" value="Short Range Force"/>
        <Parameter name="Contact Radius" type="double" value="{contact_radius:.6e}"/>
        <Parameter name="Spring Constant" type="double" value="1.0e13"/>
      </ParameterList>
    </ParameterList>
    <ParameterList name="Interactions">
      <ParameterList name="Interaction Bullet Mug">
        <Parameter name="First Block" type="string" value="{bullet_block}"/>
        <Parameter name="Second Block" type="string" value="{mug_block}"/>
        <Parameter name="Contact Model" type="string" value="Bullet Mug Contact"/>
      </ParameterList>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Boundary Conditions">
    <ParameterList name="Gravity Mug">
      <Parameter name="Type" type="string" value="Body Force"/>
      <Parameter name="Node Set" type="string" value="{mug_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{-mug_density * gravity:.6e}"/>
    </ParameterList>
    <ParameterList name="Gravity Bullet">
      <Parameter name="Type" type="string" value="Body Force"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{-bullet_density * gravity:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity X">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="x"/>
      <Parameter name="Value" type="string" value="{vx:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity Y">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="y"/>
      <Parameter name="Value" type="string" value="{vy:.6e}"/>
    </ParameterList>
    <ParameterList name="Bullet Velocity Z">
      <Parameter name="Type" type="string" value="Initial Velocity"/>
      <Parameter name="Node Set" type="string" value="{bullet_nodeset}"/>
      <Parameter name="Coordinate" type="string" value="z"/>
      <Parameter name="Value" type="string" value="{vz:.6e}"/>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Solver">
    <Parameter name="Verbose" type="bool" value="{str(verbose).lower()}"/>
    <Parameter name="Initial Time" type="double" value="0.0"/>
    <Parameter name="Final Time" type="double" value="{final_time:.6e}"/>
    <ParameterList name="Verlet">
      <Parameter name="Fixed dt" type="double" value="{dt:.6e}"/>
    </ParameterList>
  </ParameterList>
  <ParameterList name="Output">
    <Parameter name="Output File Type" type="string" value="ExodusII"/>
    <Parameter name="Output Filename" type="string" value="{Path(mesh_file).stem}"/>
    <Parameter name="Output Frequency" type="int" value="{output_frequency}"/>
    <ParameterList name="Output Variables">
      <Parameter name="Displacement" type="bool" value="true"/>
      <Parameter name="Velocity" type="bool" value="true"/>
      <Parameter name="Element_Id" type="bool" value="true"/>
      <Parameter name="Proc_Num" type="bool" value="true"/>
      <Parameter name="Dilatation" type="bool" value="true"/>
      <Parameter name="Weighted_Volume" type="bool" value="true"/>
      <Parameter name="Volume" type="bool" value="true"/>
      <Parameter name="Force" type="bool" value="true"/>
      <Parameter name="Contact_Force" type="bool" value="true"/>
      <Parameter name="Number_Of_Neighbors" type="bool" value="true"/>
      <Parameter name="Radius" type="bool" value="true"/>
      <Parameter name="Coordinates" type="bool" value="true"/>
      <Parameter name="Force_Density" type="bool" value="true"/>
      <Parameter name="Kinetic_Energy" type="bool" value="true"/>
      <Parameter name="Damage" type="bool" value="true"/>
    </ParameterList>
  </ParameterList>
</ParameterList>
'''
    if output_xml is not None:
        Path(output_xml).write_text(xml, encoding="utf-8")
    return xml

In [26]:
# ---- Bullet Vase No Floor ----
info = build_bullet_vase_nf_scene(seed=99, max_nodes=50000)

mesh_out = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\bullet_vase_nf_test.g"
cubit.cmd(f'export mesh "{mesh_out}" dimension 3 overwrite')

xml_out = mesh_out.replace(".g", ".xml")
xml = generate_bullet_vase_nf_peridigm_xml(
    mesh_file=os.path.basename(mesh_out),
    info=info,
    output_xml=xml_out,
    bullet_speed=random.uniform(100, 300),
)

log = mesh_out.replace(".g", ".log")
with open(log, "w") as f:
    for k, v in info.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nxml: {xml_out}\n")

In [27]:
# ---- Bullet Mug No Floor ----
info = build_bullet_mug_nf_scene(seed=99, max_nodes=50000)

mesh_out = r"F:\Peridigm-pre-process\dynamic-impact-generation\files\bullet_mug_nf_test.g"
cubit.cmd(f'export mesh "{mesh_out}" dimension 3 overwrite')

xml_out = mesh_out.replace(".g", ".xml")
xml = generate_bullet_mug_nf_peridigm_xml(
    mesh_file=os.path.basename(mesh_out),
    info=info,
    output_xml=xml_out,
    bullet_speed=random.uniform(100, 300),
)

log = mesh_out.replace(".g", ".log")
with open(log, "w") as f:
    for k, v in info.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nxml: {xml_out}\n")